In [ ]:
# Import Required Libraries
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge

plt.style.use("ggplot")

# Load California Housing Dataset
print("="*60)
print("Loading California Housing Dataset")
print("="*60)

housing = fetch_california_housing(as_frame=True)
df = housing.frame
df.rename(
    columns={
        "MedHouseVal":"HousePrice"
    },
    inplace=True
)

print("\nFirst Five Rows\n")
print(df.head())

# Dataset Overview
print("\nDataset Shape")
print(df.shape)
print("\nColumn Names")
print(df.columns.tolist())
print("\nMissing Values")
print(df.isnull().sum())
print("\nSummary Statistics")
print(df.describe().round(2))

# Exploratory Data Analysis
# House Price Distribution
plt.figure(figsize=(8,5))

plt.hist(
    df["HousePrice"],
    bins=30,
    edgecolor="black"
)

plt.title("Distribution of House Prices")
plt.xlabel("House Price")
plt.ylabel("Frequency")
plt.show()

#Population vs House Price
plt.figure(figsize=(8,5))

plt.scatter(
    df["Population"],
    df["HousePrice"],
    alpha=0.4
)

plt.title("Population vs House Price")
plt.xlabel("Population")
plt.ylabel("House Price")
plt.show()

# Median Income Distribution
plt.figure(figsize=(8,5))

sns.boxplot(
    x=df["MedInc"]
)

plt.title("Median Income Distribution")
plt.show()

# Separate Features and Target
X = df.drop(
    "HousePrice",
    axis=1
)

y = df["HousePrice"]
print("\nFeature Matrix :", X.shape)
print("Target Vector :", y.shape)

# Feature Scaling
print("\nApplying StandardScaler...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaling Completed Successfully")

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining Samples :", len(X_train))
print("Testing Samples :", len(X_test))

# Detect Overfitting
print("\nChecking for Overfitting...\n")

baseline_tree = DecisionTreeRegressor(
    random_state=42
)

baseline_tree.fit(
    X_train,
    y_train
)

train_prediction = baseline_tree.predict(
    X_train
)

test_prediction = baseline_tree.predict(
    X_test
)

train_rmse = np.sqrt(
    mean_squared_error(
        y_train,
        train_prediction
    )
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test_prediction
    )
)

print("Training RMSE :", round(train_rmse,4))
print("Testing RMSE  :", round(test_rmse,4))

if test_rmse > train_rmse:
    print("\nObservation")
    print("Model shows signs of overfitting.")

else:
    print("\nObservation")
    print("No major overfitting detected.")

# Train vs Test RMSE
comparison = pd.DataFrame({
    "Dataset":[
        "Training",
        "Testing"
    ],
    "RMSE":[
        train_rmse,
        test_rmse
    ]
})

plt.figure(figsize=(6,4))

plt.bar(
    comparison["Dataset"],
    comparison["RMSE"]
)

plt.title("Training vs Testing RMSE")
plt.ylabel("RMSE")
plt.show()

# Cross Validation & Hyperparameter Tuning
print("\n" + "="*60)
print("BASELINE MODEL TRAINING")
print("="*60)

# Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_prediction = linear_model.predict(X_test)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_prediction
    )
)

linear_r2 = r2_score(
    y_test,
    linear_prediction
)

print("\nLinear Regression")
print("RMSE :", round(linear_rmse,4))
print("R² Score :", round(linear_r2,4))

# Ridge Regression
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)
ridge_prediction = ridge_model.predict(X_test)

ridge_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        ridge_prediction
    )
)

ridge_r2 = r2_score(
    y_test,
    ridge_prediction
)

print("\nRidge Regression")
print("RMSE :", round(ridge_rmse,4))
print("R² Score :", round(ridge_r2,4))

# Cross Validation
print("\n" + "="*60)
print("5-FOLD CROSS VALIDATION")
print("="*60)

cv_scores = cross_val_score(
    baseline_tree,
    X_scaled,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

cv_rmse = -cv_scores
print("\nRMSE for Each Fold")

for i, score in enumerate(cv_rmse):
    print(f"Fold {i+1} : {score:.4f}")

print("\nAverage RMSE :", round(cv_rmse.mean(),4))
print("Standard Deviation :", round(cv_rmse.std(),4))

# Hyperparameter Tuning
print("\n" + "="*60)
print("GRID SEARCH")
print("="*60)

parameter_grid = {
    "max_depth":[
        3,
        5,
        7,
        10
    ],
    "min_samples_split":[
        2,
        5,
        10
    ]
}

grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=parameter_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(
    X_train,
    y_train
)

print("\nBest Parameters")
print(grid_search.best_params_)
print("\nBest Cross Validation Score")
print(round(-grid_search.best_score_,4))

# Optimized Model
best_tree = grid_search.best_estimator_
best_prediction = best_tree.predict(X_test)

tree_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        best_prediction
    )
)

tree_r2 = r2_score(
    y_test,
    best_prediction
)

print("\nOptimized Decision Tree")
print("RMSE :", round(tree_rmse,4))
print("R² Score :", round(tree_r2,4))

# Performance Visualization
performance = pd.DataFrame({
    "Metric":[
        "Train RMSE",
        "Test RMSE",
        "Cross Validation"
    ],
    "Value":[
        train_rmse,
        test_rmse,
        cv_rmse.mean()
    ]
})

plt.figure(figsize=(7,4))

plt.plot(
    performance["Metric"],
    performance["Value"],
    marker="o",
    linewidth=2
)

plt.title("Model Performance Overview")
plt.ylabel("RMSE")
plt.grid(True)
plt.show()

# Final Model Comparison & Selection
print("\n" + "="*65)
print("MODEL COMPARISON")
print("="*65)

# Create Comparison Table
comparison_df = pd.DataFrame({
    "Model":[
        "Linear Regression",
        "Ridge Regression",
        "Optimized Decision Tree"
    ],
    "RMSE":[
        linear_rmse,
        ridge_rmse,
        tree_rmse
    ],
    "R2 Score":[
        linear_r2,
        ridge_r2,
        tree_r2
    ]
})

comparison_df = comparison_df.sort_values(
    by="R2 Score",
    ascending=False
).reset_index(drop=True)

comparison_df.insert(
    0,
    "Rank",
    range(1, len(comparison_df)+1)
)

print(comparison_df.round(4))

# Best Model
best_model_name = comparison_df.loc[0,"Model"]
print("\nBest Model Selected :", best_model_name)

# RMSE Comparison

plt.figure(figsize=(8,5))

plt.bar(
    comparison_df["Model"],
    comparison_df["RMSE"]
)

plt.ylabel("RMSE")
plt.title("RMSE Comparison of Regression Models")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# R² Comparison

plt.figure(figsize=(8,5))

plt.bar(
    comparison_df["Model"],
    comparison_df["R2 Score"]
)

plt.ylabel("R² Score")
plt.title("R² Score Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Actual vs Predicted

plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    best_prediction,
    alpha=0.5
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red",
    linestyle="--"
)

plt.xlabel("Actual House Price")
plt.ylabel("Predicted House Price")
plt.title("Optimized Decision Tree Predictions")
plt.grid(True)
plt.show()

# Feature Importance
importance = pd.DataFrame({
    "Feature":X.columns,
    "Importance":best_tree.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

plt.figure(figsize=(8,5))

plt.bar(
    importance["Feature"],
    importance["Importance"]
)

plt.xticks(rotation=45)
plt.ylabel("Importance")
plt.title("Feature Importance")
plt.tight_layout()
plt.show()
print("\nFeature Importance")
print(importance.round(3))

# Save Optimized Model
joblib.dump(
    best_tree,
    "optimized_house_price_model.pkl"
)

print("\nOptimized model saved successfully!")
print("Filename : optimized_house_price_model.pkl")

# Predict New House Price
sample_house = pd.DataFrame(
    [[
        8.32,
        41,
        6.98,
        1.02,
        322,
        2.55,
        37.88,
        -122.23
    ]],
    columns=X.columns
)

sample_scaled = scaler.transform(sample_house)
future_price = best_tree.predict(sample_scaled)

print("\nPredicted House Price")
print(round(future_price[0],3))

# Final Justification
print("\n" + "="*65)
print("FINAL MODEL JUSTIFICATION")
print("="*65)

print("""

1. Cross-validation was used to obtain a more reliable estimate
   of model performance instead of relying on a single train-test split.

2. Hyperparameter tuning using GridSearchCV improved the
   Decision Tree by selecting suitable values for max_depth
   and min_samples_split.

3. The optimized Decision Tree achieved better generalization
   and reduced overfitting compared to the baseline model.

4. Final model selection was based on higher R² Score and
   lower RMSE after tuning.

""")

# Project Summary
summary = pd.DataFrame({
    "Category":[
        "Dataset",
        "Scaling",
        "Validation",
        "Hyperparameter Tuning",
        "Best Model",
        "Saved Model"
    ],

    "Details":[
        "California Housing",
        "StandardScaler",
        "5-Fold Cross Validation",
        "GridSearchCV",
        best_model_name,
        "optimized_house_price_model.pkl"
    ]
})

print(summary.to_string(index=False))

